# Data Preparation

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight

In [5]:
!wget -O data_balita.csv https://raw.githubusercontent.com/azzaauliyaul/Detection-of-Stunting-in-Indonesian-Toddlers/main/data/data_balita.csv

--2026-11-17 13:27:28--  https://raw.githubusercontent.com/azzaauliyaul/Detection-of-Stunting-in-Indonesian-Toddlers/main/data/data_balita.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3299166 (3.1M) [text/plain]
Saving to: ‘data_balita.csv’

data_balita.csv     100%[===================>]   3.15M  14.7MB/s    in 0.2s    

2026-11-17 13:27:29 (14.7 MB/s) - ‘data_balita.csv’ saved [3299166/3299166]



In [6]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

data = pd.read_csv('data_balita.csv')

data['Jenis Kelamin'] = data['Jenis Kelamin'].str.strip().str.lower()
data['Status Gizi'] = data['Status Gizi'].str.strip().str.lower()
data = data.drop_duplicates().reset_index(drop=True)

X = data.drop('Status Gizi', axis=1).copy()
y = data['Status Gizi'].copy()

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30,
    stratify=y,
    random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50,
    stratify=y_temp,
    random_state=RANDOM_STATE
)

In [7]:
gender_encoder = LabelEncoder()
X_train['Jenis Kelamin'] = gender_encoder.fit_transform(X_train['Jenis Kelamin'])
X_val['Jenis Kelamin'] = gender_encoder.transform(X_val['Jenis Kelamin'])
X_test['Jenis Kelamin'] = gender_encoder.transform(X_test['Jenis Kelamin'])

target_encoder = LabelEncoder()
y_train = target_encoder.fit_transform(y_train)
y_val = target_encoder.transform(y_val)
y_test = target_encoder.transform(y_test)

num_cols = ["Umur (bulan)", "Tinggi Badan (cm)"]

scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_val[num_cols] = scaler.transform(X_val[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weights = {int(c): float(w) for c, w in zip(classes, weights)}
sample_weights_train = compute_sample_weight(class_weight="balanced", y=y_train)

In [8]:
target_names = list(target_encoder.classes_)
print(f"Data bersih: {len(data):,} | Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")
print("Gender mapping:", dict(zip(gender_encoder.classes_, gender_encoder.transform(gender_encoder.classes_))))
print("Target mapping:", dict(enumerate(target_names)))
print("Class weights :", {k: round(v, 4) for k, v in class_weights.items()})

Data bersih: 39,425 | Train: 27,597 | Val: 5,914 | Test: 5,914
Gender mapping: {'laki-laki': np.int64(0), 'perempuan': np.int64(1)}
Target mapping: {0: 'normal', 1: 'severely stunted', 2: 'stunted', 3: 'tinggi'}
Class weights : {0: 0.4581, 1: 1.5117, 2: 2.2313, 3: 1.4132}


In [ ]:
import joblib
from google.colab import files

ARTIFACT_PATH = "preprocessed_stunting_data.joblib"

artifacts = {
    "X_train": X_train,
    "X_val": X_val,
    "X_test": X_test,
    "y_train": y_train,
    "y_val": y_val,
    "y_test": y_test,
    "gender_encoder": gender_encoder,
    "target_encoder": target_encoder,
    "scaler": scaler,
    "class_weights": class_weights,
    "random_state": RANDOM_STATE,
}

joblib.dump(artifacts, ARTIFACT_PATH)
print(f"File berhasil dibuat: {ARTIFACT_PATH}")

files.download(ARTIFACT_PATH)